In [20]:
# Automatic single-folder/picture predictor with robust autocrop + CTC decode
import os, numpy as np, tensorflow as tf
from PIL import Image, ImageDraw, ImageFont
import cv2

# ---- configure these paths ----
MODEL_PATH = r"C:\Users\aswin\Desktop\Finalyearproject\check2\best_ctc_inference_2.keras"
CHARSET_PATH = r"C:\Users\aswin\Desktop\Finalyearproject\check2\charset.txt"   # one token per line (in training order)

# Either point IMG_PATH to a single file or a folder. If folder, all images will be processed.
IMG_PATH = r"C:\Users\aswin\Desktop\Finalyearproject\check2\download.jpg"   # or r"path\to\folder_with_images"
IMG_HEIGHT = 64
MAX_IMG_WIDTH = 640
BEAM = 1   # 1 = greedy; >1 uses beam search (beam may be fragile)

# ---- helpers ----
def load_charset(path_or_literal):
    if os.path.exists(path_or_literal):
        toks = []
        with open(path_or_literal, "r", encoding="utf-8") as f:
            for line in f:
                s = line.rstrip("\n\r")
                if s == "": continue
                toks.append(s)
    else:
        toks = list(path_or_literal)
    num_to_char = {i: c for i, c in enumerate(toks)}
    return num_to_char, toks

def preprocess_image_arr(pil_img, img_height=IMG_HEIGHT, max_width=MAX_IMG_WIDTH):
    """Resize (preserve aspect), normalize, pad/truncate to max_width and return arr and resized width."""
    img = pil_img.convert("L")
    w,h = img.size
    new_h = img_height
    new_w = max(1, int(w * (new_h/float(h))))
    img = img.resize((new_w, new_h), Image.LANCZOS)
    arr = np.array(img).astype(np.float32) / 255.0
    arr = arr * 2.0 - 1.0
    if new_w < max_width:
        pad = np.ones((new_h, max_width), dtype=np.float32) * (-1.0)
        pad[:, :new_w] = arr
        arr = pad
    elif new_w > max_width:
        arr = arr[:, :max_width]
        new_w = max_width
    arr = np.expand_dims(arr, -1)
    return arr, new_w

def map_ids_to_text(ids_row, logits_or_probs, num_to_char):
    blank = logits_or_probs.shape[-1] - 1
    out = []
    for cid in ids_row:
        if int(cid) == -1:
            continue
        if int(cid) >= blank:
            continue
        out.append(num_to_char.get(int(cid), "?"))
    return "".join(out)

def manual_argmax_collapse(logits_np, num_to_char):
    arg = np.argmax(logits_np, axis=-1)[0].tolist()
    blank = logits_np.shape[-1] - 1
    collapsed = []
    prev = None
    for a in arg:
        if a == prev:
            prev = a
            continue
        if a == blank:
            prev = a
            continue
        collapsed.append(a)
        prev = a
    text = "".join([num_to_char.get(int(i), "?") for i in collapsed])
    return text, arg, collapsed

# ---- robust autocrop (projection-based) ----
def autocrop_braille_pil(pil_img, dark_thresh=240, pad=10):
    """Given a PIL Image, return a cropped PIL Image containing the braille region.
       dark_thresh: pixels darker than this are considered 'dot' (0-255)
    """
    gray = np.array(pil_img.convert("L"))
    # Contrast normalize
    gray = cv2.normalize(gray, None, 0, 255, cv2.NORM_MINMAX)
    # mask of dark pixels (True for dark)
    mask = gray < dark_thresh
    col_sum = np.sum(mask, axis=0)
    row_sum = np.sum(mask, axis=1)
    cols = np.where(col_sum > 0)[0]
    rows = np.where(row_sum > 0)[0]
    if len(cols) == 0 or len(rows) == 0:
        # fallback: try a looser threshold
        mask2 = gray < (dark_thresh + 20)
        cols = np.where(np.sum(mask2, axis=0) > 0)[0]
        rows = np.where(np.sum(mask2, axis=1) > 0)[0]
        if len(cols) == 0 or len(rows) == 0:
            # nothing detected; return original
            return pil_img.copy()
    x1, x2 = int(cols[0]), int(cols[-1])
    y1, y2 = int(rows[0]), int(rows[-1])
    h, w = gray.shape
    x1 = max(0, x1 - pad)
    y1 = max(0, y1 - pad)
    x2 = min(w, x2 + pad)
    y2 = min(h, y2 + pad)
    return pil_img.crop((x1, y1, x2, y2))

# ---- load artifacts ----
print("Loading model:", MODEL_PATH)
model = tf.keras.models.load_model(MODEL_PATH, compile=False)
print("Loaded. model output classes:", model.output_shape[-1])
num_to_char, toks = load_charset(CHARSET_PATH)
print("Loaded charset tokens:", len(toks), "(0..{})".format(len(toks)-1))
print("First tokens:", toks[:20])

# ---- main processing function ----
def process_and_predict(image_path, save_overlay=True, out_dir=None):
    if out_dir is None:
        out_dir = os.path.dirname(image_path) or os.getcwd()
    os.makedirs(out_dir, exist_ok=True)

    pil = Image.open(image_path)
    # autocrop
    cropped = autocrop_braille_pil(pil, dark_thresh=240, pad=10)
    base = os.path.splitext(os.path.basename(image_path))[0]
    cropped_path = os.path.join(out_dir, base + "_cropped.png")
    cropped.save(cropped_path)
    # preprocess
    arr, w = preprocess_image_arr(cropped, IMG_HEIGHT, MAX_IMG_WIDTH)
    X = np.expand_dims(arr, 0).astype(np.float32)
    # predict logits
    logits = model.predict(X)  # shape (1, T, C)
    # probs
    probs = tf.nn.softmax(logits, axis=-1).numpy()

    # decode using probs (preferred)
    text_probs = ""
    try:
        decoded_probs = tf.keras.backend.ctc_decode(
            probs, input_length=np.ones((probs.shape[0],), dtype=np.int32)*probs.shape[1],
            greedy=(BEAM==1), beam_width=BEAM, top_paths=1
        )
        first = decoded_probs[0][0]
        if isinstance(first, tf.SparseTensor):
            dense_probs = tf.sparse.to_dense(first, default_value=-1).numpy()
        elif tf.is_tensor(first):
            dense_probs = first.numpy()
        else:
            dense_probs = np.asarray(first)
        text_probs = map_ids_to_text(dense_probs[0], probs, num_to_char)
    except Exception as e:
        print("ctc_decode on PROBS failed:", e)
        text_probs = ""

    # decode using logits (debug)
    text_logits = ""
    try:
        decoded_logits = tf.keras.backend.ctc_decode(
            logits, input_length=np.ones((logits.shape[0],), dtype=np.int32)*logits.shape[1],
            greedy=(BEAM==1), beam_width=BEAM, top_paths=1
        )
        first2 = decoded_logits[0][0]
        if isinstance(first2, tf.SparseTensor):
            dense_logits = tf.sparse.to_dense(first2, default_value=-1).numpy()
        elif tf.is_tensor(first2):
            dense_logits = first2.numpy()
        else:
            dense_logits = np.asarray(first2)
        text_logits = map_ids_to_text(dense_logits[0], logits, num_to_char)
    except Exception as e:
        print("ctc_decode on LOGITS failed:", e)
        text_logits = ""

    text_manual, argseq, collapsed_ids = manual_argmax_collapse(logits, num_to_char)

    final = text_probs if (text_probs is not None and len(text_probs)>0) else text_manual
    print(f"\nImage: {image_path}")
    print(" -> Cropped saved:", cropped_path)
    print("CTC greedy (on PROBS):", repr(text_probs))
    print("CTC greedy (on LOGITS):", repr(text_logits))
    print("Manual argmax-collapse:", repr(text_manual))
    print("FINAL:", repr(final))

    if save_overlay:
        arr_disp = ((arr.squeeze() + 1.0)/2.0*255.0).clip(0,255).astype(np.uint8)
        pil_disp = Image.fromarray(arr_disp).convert("RGB")
        draw = ImageDraw.Draw(pil_disp)
        try:
            font = ImageFont.truetype("DejaVuSans-Bold.ttf", 18)
        except Exception:
            font = ImageFont.load_default()
        draw.text((4,4), f"PRED: {final}", fill=(255,0,0), font=font)
        overlay_path = os.path.join(out_dir, base + "_overlay.png")
        pil_disp.save(overlay_path)
        print("Overlay saved:", overlay_path)

    return {
        "image": image_path,
        "cropped": cropped_path,
        "pred_probs": text_probs,
        "pred_logits": text_logits,
        "pred_manual": text_manual,
        "final": final
    }

Loading model: C:\Users\aswin\Desktop\Finalyearproject\check2\best_ctc_inference_2.keras
Loaded. model output classes: 27
Loaded charset tokens: 26 (0..25)
First tokens: ['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T']


In [21]:
# run in python
import tensorflow as tf
print('tf version', tf.__version__)
print('devices', tf.config.list_physical_devices())
# ---- convenience: process single-file or folder ----
results = []
if os.path.isdir(IMG_PATH):
    for fname in sorted(os.listdir(IMG_PATH)):
        if fname.lower().endswith((".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff")):
            p = os.path.join(IMG_PATH, fname)
            results.append(process_and_predict(p, save_overlay=True, out_dir=IMG_PATH))
else:
    results.append(process_and_predict(IMG_PATH, save_overlay=True, out_dir=os.path.dirname(IMG_PATH) or "."))

# ---- final summary
print("\nProcessed:", len(results), "items.")
for r in results:
    print(r["image"], "->", r["final"])

tf version 2.19.0
devices [PhysicalDevice(name='/physical_device:CPU:0', device_type='CPU')]
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 533ms/step

Image: C:\Users\aswin\Desktop\Finalyearproject\check2\download.jpg
 -> Cropped saved: C:\Users\aswin\Desktop\Finalyearproject\check2\download_cropped.png
CTC greedy (on PROBS): 'A'
CTC greedy (on LOGITS): 'AA'
Manual argmax-collapse: 'A'
FINAL: 'A'
Overlay saved: C:\Users\aswin\Desktop\Finalyearproject\check2\download_overlay.png

Processed: 1 items.
C:\Users\aswin\Desktop\Finalyearproject\check2\download.jpg -> A
